<a href="https://colab.research.google.com/github/morrita10305/ML_final/blob/main/ai_f1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# PART 1 — Environment Setup (環境與套件載入)
# ============================================================

import os
import gc
import random
import warnings

# 隱藏警告訊息，保持輸出乾淨
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# PyTorch 核心與模型建構套件
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

# 資料預處理與模型評估指標
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# F1 賽車數據 API
import fastf1

# ============================================================
# Reproducibility (設定隨機種子以確保實驗可重複)
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# 強制 cuDNN 使用固定演算法，確保每次執行的準確率/損失完全一致
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ============================================================
# Device (配置硬體加速：優先使用 GPU，若無則用 CPU)
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(f"Device: {device}")

if torch.cuda.is_available():
    print(
        f"GPU: {torch.cuda.get_device_name(0)}"
    )

# ============================================================
# FastF1 Cache (啟用快取避免重複下載賽事資料)
# ============================================================

CACHE_DIR = "./fastf1_cache"

os.makedirs(
    CACHE_DIR,
    exist_ok=True
)

# 啟用快取機制，大幅提升後續讀取賽車數據的速度
fastf1.Cache.enable_cache(
    CACHE_DIR
)

print("FastF1 Cache Enabled")

Device: cuda
GPU: NVIDIA RTX A2000 12GB
FastF1 Cache Enabled


In [ ]:
# ============================================================
# PART 2 — Auto Discover Sessions
# Research Version
# ============================================================

START_YEAR = 2019
END_YEAR = 2024

all_sessions = []

session_count = 0

for year in range(
        START_YEAR,
        END_YEAR + 1
):

    print(f"\nLoading Schedule {year}")

    try:

        schedule = fastf1.get_event_schedule(
            year
        )

    except Exception as e:

        print(
            f"Schedule Error {year}: {e}"
        )

        continue

    for _, event in schedule.iterrows():

        try:

            race_name = event["EventName"]

            round_number = event.get(
                "RoundNumber",
                np.nan
            )

            event_date = event.get(
                "EventDate",
                pd.NaT
            )

            print(
                f"Checking {year} {race_name}"
            )

            session = fastf1.get_session(
                year,
                race_name,
                "R"
            )

            session.load(
                laps=False,
                telemetry=False,
                weather=False
            )

            if session.results is None:
                continue

            if len(session.results) == 0:
                continue

            drivers = (

                session.results[
                    "Abbreviation"
                ]

                .dropna()

                .unique()

                .tolist()

            )

            drivers = sorted(
                drivers
            )

            if len(drivers) < 10:
                continue

            all_sessions.append({

                "year": int(year),

                "round": int(round_number),

                "event_date": event_date,

                "race": race_name,

                "location":
                    getattr(
                        session.event,
                        "Location",
                        None
                    ),

                "country":
                    getattr(
                        session.event,
                        "Country",
                        None
                    ),

                "event_format":
                    session.event.EventFormat,

                "total_drivers":
                    len(drivers),

                "drivers":
                    drivers

            })

            session_count += 1

            print(
                f"Drivers Found: {len(drivers)}"
            )

        except Exception as e:

            print(
                f"Session Skip: {e}"
            )

print()
print(
    f"Total Sessions: {session_count}"
)
print()

if len(all_sessions) > 0:

    print("Example Session:")

    print(
        all_sessions[0]
    )


Loading Schedule 2019


core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


Checking 2019 Australian Grand Prix


req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['77', '44', '33', '5', '16', '20', '27', '7', '18', '26', '10', '4', '11', '23', '99', '63', '88', '8', '3', '55']
core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


Drivers Found: 20
Checking 2019 Bahrain Grand Prix


...

Drivers Found: 20
Checking 2024 Qatar Grand Prix


Request for URL https://api.jolpi.ca/ergast/f1/2024/23/results.json failed; using cached response
Traceback (most recent call last):
  File "C:\Users\leow3\anaconda3\envs\pytorch_gpu\Lib\site-packages\requests_cache\session.py", line 316, in _resend
    response.raise_for_status()
  File "C:\Users\leow3\anaconda3\envs\pytorch_gpu\Lib\site-packages\requests\models.py", line 1028, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 429 Client Error: Too Many Requests for url: https://api.jolpi.ca/ergast/f1/2024/23/results.json
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '16', '81', '63', '10', '55', '14', '24', '20', '4', '77', '44', '22', '30', '23', '27', '11', '18', '43', '31']
core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for dr

Drivers Found: 20
Checking 2024 Abu Dhabi Grand Prix


Request for URL https://api.jolpi.ca/ergast/f1/2024/24/results.json failed; using cached response
Traceback (most recent call last):
  File "C:\Users\leow3\anaconda3\envs\pytorch_gpu\Lib\site-packages\requests_cache\session.py", line 316, in _resend
    response.raise_for_status()
  File "C:\Users\leow3\anaconda3\envs\pytorch_gpu\Lib\site-packages\requests\models.py", line 1028, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 429 Client Error: Too Many Requests for url: https://api.jolpi.ca/ergast/f1/2024/24/results.json
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '55', '16', '44', '63', '1', '10', '27', '14', '81', '23', '22', '24', '18', '61', '20', '30', '77', '43', '11']


Drivers Found: 20

Total Sessions: 135

Example Session:
{'year': 2019, 'round': 1, 'event_date': Timestamp('2019-03-17 00:00:00'), 'race': 'Australian Grand Prix', 'location': 'Melbourne', 'country': 'Australia', 'event_format': 'conventional', 'total_drivers': 20, 'drivers': ['ALB', 'BOT', 'GAS', 'GIO', 'GRO', 'HAM', 'HUL', 'KUB', 'KVY', 'LEC', 'MAG', 'NOR', 'PER', 'RAI', 'RIC', 'RUS', 'SAI', 'STR', 'VER', 'VET']}


In [ ]:
# ============================================================
# PART 3 - Physics Proxy Functions
# Research-grade, leakage-aware feature proxies
# ============================================================

import math
import numpy as np
import pandas as pd


FIA_MAX_RACE_FUEL_KG = 110.0


def _clip(value, lower, upper, default=0.0):
    try:
        value = float(value)
        if not np.isfinite(value):
            return float(default)
        return float(np.clip(value, lower, upper))
    except Exception:
        return float(default)


def _as_float(value, default=0.0):
    try:
        if pd.isna(value):
            return float(default)
        value = float(value)
        if not np.isfinite(value):
            return float(default)
        return value
    except Exception:
        return float(default)


def safe_seconds(value, default=np.nan):
    """Convert pandas/stdlib timedeltas and numeric values to seconds."""
    try:
        if pd.isna(value):
            return default
        if hasattr(value, "total_seconds"):
            value = value.total_seconds()
        value = float(value)
        if not np.isfinite(value):
            return default
        return value
    except Exception:
        return default


def safe_mean(values, default=0.0):
    """NaN-safe mean for pandas Series, numpy arrays, lists, and scalars."""
    try:
        series = pd.Series(values)
        series = pd.to_numeric(series, errors="coerce")
        series = series.replace([np.inf, -np.inf], np.nan).dropna()
        if len(series) == 0:
            return float(default)
        return float(series.mean())
    except Exception:
        return float(default)


def safe_std(values, default=0.0):
    """Population standard deviation with robust NaN handling."""
    try:
        series = pd.Series(values)
        series = pd.to_numeric(series, errors="coerce")
        series = series.replace([np.inf, -np.inf], np.nan).dropna()
        if len(series) <= 1:
            return float(default)
        return float(series.std(ddof=0))
    except Exception:
        return float(default)


def calculate_lap_ratio(lap_number, total_laps):
    """Current race progress in [0, 1]."""
    lap_number = _as_float(lap_number, 0.0)
    total_laps = max(_as_float(total_laps, 0.0), 1.0)
    return _clip(lap_number / total_laps, 0.0, 1.0)


def estimate_fuel_mass(lap_number, total_laps, race_fuel_kg=FIA_MAX_RACE_FUEL_KG):
    """
    FIA fuel-mass proxy.

    Formula uses the 110 kg race fuel limit and a monotonic burn-down curve.
    Lap 1 starts near full fuel by using lap_number - 1. This is a proxy,
    not measured fuel mass, and does not use future timing information.
    """
    lap_number = max(_as_float(lap_number, 1.0), 1.0)
    total_laps = max(_as_float(total_laps, 1.0), 1.0)
    race_fuel_kg = max(_as_float(race_fuel_kg, FIA_MAX_RACE_FUEL_KG), 1.0)
    completed_ratio = _clip((lap_number - 1.0) / total_laps, 0.0, 1.0)
    fuel_mass = race_fuel_kg * (1.0 - completed_ratio)
    return _clip(fuel_mass, 0.0, race_fuel_kg, default=race_fuel_kg)


def estimate_fuel_load(lap_number, total_laps, race_fuel=FIA_MAX_RACE_FUEL_KG):
    """Backward-compatible alias for older notebook cells."""
    return estimate_fuel_mass(lap_number, total_laps, race_fuel)


def expected_tyre_life(compound):
    """Approximate dry/wet tyre working-life prior in laps."""
    compound = str(compound).upper()
    if "SOFT" in compound:
        return 18.0
    if "MEDIUM" in compound:
        return 26.0
    if "HARD" in compound:
        return 36.0
    if "INTERMEDIATE" in compound:
        return 20.0
    if "WET" in compound:
        return 24.0
    return 28.0


def tire_temp_proxy(
    speed_mean,
    throttle_mean,
    brake_mean,
    track_temp,
    tyre_life=0,
    compound="UNKNOWN",
    rainfall=0,
    humidity=0,
):
    """
    Tire carcass/surface temperature proxy.

    The model follows the qualitative structure of common racing thermal
    models: track temperature baseline, speed/friction heating, traction
    heating, brake-energy heating, age/wear heat retention, and wet cooling.
    It is deliberately bounded to plausible F1 tyre operating ranges.
    """
    compound = str(compound).upper()
    speed_norm = _clip(speed_mean, 0.0, 360.0) / 360.0
    throttle_norm = _clip(throttle_mean, 0.0, 100.0) / 100.0

    brake_value = _as_float(brake_mean, 0.0)
    brake_norm = brake_value if brake_value <= 1.0 else brake_value / 100.0
    brake_norm = _clip(brake_norm, 0.0, 1.0)

    track_temp = _clip(track_temp, -5.0, 80.0, default=30.0)
    tyre_life = max(_as_float(tyre_life, 0.0), 0.0)
    age_norm = _clip(tyre_life / expected_tyre_life(compound), 0.0, 1.8)

    humidity_norm = _clip(humidity, 0.0, 100.0) / 100.0
    wet_flag = 1.0 if bool(rainfall) or "INTERMEDIATE" in compound or compound == "WET" else 0.0

    compound_offset = 0.0
    if "SOFT" in compound:
        compound_offset = 4.0
    elif "MEDIUM" in compound:
        compound_offset = 1.0
    elif "HARD" in compound:
        compound_offset = -2.0
    elif "INTERMEDIATE" in compound:
        compound_offset = -8.0
    elif "WET" in compound:
        compound_offset = -12.0

    heat_input = (
        42.0 * speed_norm
        + 18.0 * throttle_norm
        + 22.0 * brake_norm
        + 7.0 * age_norm
        + compound_offset
    )
    wet_cooling = wet_flag * (10.0 + 8.0 * humidity_norm)
    return _clip(track_temp + heat_input - wet_cooling, 35.0, 145.0)


def _clean_lap_history(frame):
    if frame is None or len(frame) == 0:
        return pd.DataFrame()

    history = frame.copy()
    if "LapTime" not in history.columns:
        return pd.DataFrame()

    history = history[history["LapTime"].notna()].copy()

    for pit_col in ["PitInTime", "PitOutTime"]:
        if pit_col in history.columns:
            history = history[history[pit_col].isna()]

    history["LapTimeSeconds"] = history["LapTime"].apply(safe_seconds)
    history = history[np.isfinite(history["LapTimeSeconds"])]
    return history


def _historical_laps(lap, laps_driver, same_stint=True, include_current=True):
    try:
        lap_number = _as_float(lap.get("LapNumber"), np.nan)
        if not np.isfinite(lap_number):
            return pd.DataFrame()

        history = laps_driver.copy()
        if include_current:
            history = history[history["LapNumber"] <= lap_number]
        else:
            history = history[history["LapNumber"] < lap_number]

        if same_stint and "Stint" in history.columns:
            stint = lap.get("Stint", np.nan)
            if not pd.isna(stint):
                history = history[history["Stint"] == stint]

        return _clean_lap_history(history).sort_values("LapNumber")
    except Exception:
        return pd.DataFrame()


def calculate_tire_degradation(lap, laps_driver, window=3):
    """
    Recent same-stint degradation: current lap time minus the previous
    rolling baseline. Positive values mean the tyre is getting slower.
    """
    try:
        current_lap = safe_seconds(lap.get("LapTime"))
        if not np.isfinite(current_lap):
            return 0.0

        history = _historical_laps(
            lap,
            laps_driver,
            same_stint=True,
            include_current=False,
        ).tail(window)

        if len(history) < 2:
            return 0.0

        baseline = history["LapTimeSeconds"].mean()
        return _clip(current_lap - baseline, -10.0, 20.0)
    except Exception:
        return 0.0


def rolling_degradation(lap, laps_driver, window=5):
    """Same-stint rolling lap-time slope in seconds per lap."""
    try:
        history = _historical_laps(
            lap,
            laps_driver,
            same_stint=True,
            include_current=True,
        ).tail(window)

        if len(history) < 3:
            return 0.0

        y = history["LapTimeSeconds"].to_numpy(dtype=float)
        x = np.arange(len(y), dtype=float)
        slope = np.polyfit(x, y, 1)[0]
        return _clip(slope, -5.0, 5.0)
    except Exception:
        return 0.0


def calculate_lap_trend(lap, laps_driver, window=5):
    """Driver-level recent lap-time trend, independent of compound stint."""
    try:
        history = _historical_laps(
            lap,
            laps_driver,
            same_stint=False,
            include_current=True,
        ).tail(window)

        if len(history) < 3:
            return 0.0

        y = history["LapTimeSeconds"].to_numpy(dtype=float)
        x = np.arange(len(y), dtype=float)
        slope = np.polyfit(x, y, 1)[0]
        return _clip(slope, -5.0, 5.0)
    except Exception:
        return 0.0


def calculate_moving_average_lap_time(lap, laps_driver, window=3):
    """Recent same-stint moving average lap time in seconds."""
    try:
        history = _historical_laps(
            lap,
            laps_driver,
            same_stint=True,
            include_current=True,
        ).tail(window)

        if len(history) == 0:
            return safe_seconds(lap.get("LapTime"), default=0.0)

        return float(history["LapTimeSeconds"].mean())
    except Exception:
        return 0.0


def calculate_speed_std(telemetry):
    try:
        if telemetry is None or len(telemetry) == 0 or "Speed" not in telemetry.columns:
            return 0.0
        return _clip(safe_std(telemetry["Speed"]), 0.0, 140.0)
    except Exception:
        return 0.0


def calculate_brake_ratio(telemetry):
    try:
        if telemetry is None or len(telemetry) == 0 or "Brake" not in telemetry.columns:
            return 0.0

        brake = pd.to_numeric(pd.Series(telemetry["Brake"]), errors="coerce").fillna(0.0)
        threshold = 0.05 if brake.max() <= 1.0 else 5.0
        return _clip((brake > threshold).mean(), 0.0, 1.0)
    except Exception:
        return 0.0


def calculate_performance_loss_rate(lap, laps_driver, window=5):
    """
    Slope of normalized pace loss within the current stint.
    Uses only current and historical laps, never future laps.
    """
    try:
        history = _historical_laps(
            lap,
            laps_driver,
            same_stint=True,
            include_current=True,
        ).tail(window)

        if len(history) < 3:
            return 0.0

        lap_seconds = history["LapTimeSeconds"].to_numpy(dtype=float)
        best_so_far = np.minimum.accumulate(lap_seconds)
        normalized_loss = (lap_seconds - best_so_far) / np.maximum(best_so_far, 1e-6)
        x = np.arange(len(normalized_loss), dtype=float)
        slope = np.polyfit(x, normalized_loss, 1)[0]
        return _clip(slope, -0.03, 0.12)
    except Exception:
        return 0.0


def calculate_normalized_pace_loss(lap_time, best_lap):
    try:
        lap_time = safe_seconds(lap_time, default=lap_time)
        best_lap = safe_seconds(best_lap, default=best_lap)
        lap_time = _as_float(lap_time, np.nan)
        best_lap = _as_float(best_lap, np.nan)
        if not np.isfinite(lap_time) or not np.isfinite(best_lap) or best_lap <= 0:
            return 0.0
        return _clip((lap_time - best_lap) / best_lap, 0.0, 0.35)
    except Exception:
        return 0.0


def infer_is_late_stint(tyre_life, compound):
    tyre_life = max(_as_float(tyre_life, 0.0), 0.0)
    return int(tyre_life >= 0.65 * expected_tyre_life(compound))


def infer_long_run_flag(tyre_life, compound):
    tyre_life = max(_as_float(tyre_life, 0.0), 0.0)
    return int(tyre_life >= 0.80 * expected_tyre_life(compound))


def calculate_pit_window_score(
    tyre_life,
    remaining_laps,
    degradation,
    pace_loss,
    compound="UNKNOWN",
    total_laps=None,
    rainfall=0,
    humidity=0,
):
    """Bounded strategy score indicating whether a stop window is opening."""
    try:
        tyre_life = max(_as_float(tyre_life, 0.0), 0.0)
        remaining_laps = max(_as_float(remaining_laps, 0.0), 0.0)
        total_laps = max(_as_float(total_laps, remaining_laps + tyre_life), 1.0)

        age_score = _clip(tyre_life / expected_tyre_life(compound), 0.0, 1.25) / 1.25
        degradation_score = _clip((degradation + 0.75) / 5.0, 0.0, 1.0)
        pace_loss_score = _clip(pace_loss / 0.07, 0.0, 1.0)
        race_progress_score = _clip(1.0 - remaining_laps / total_laps, 0.0, 1.0)
        wet_score = _clip((float(bool(rainfall)) + (_clip(humidity, 0.0, 100.0) / 100.0)) / 2.0, 0.0, 1.0)

        score = (
            0.30 * age_score
            + 0.25 * degradation_score
            + 0.20 * pace_loss_score
            + 0.15 * race_progress_score
            + 0.10 * wet_score
        )
        return _clip(score, 0.0, 1.0)
    except Exception:
        return 0.0


def calculate_pit_pressure_index(
    pit_window_score,
    tyre_life,
    degradation,
    performance_loss_rate,
    normalized_pace_loss,
    remaining_laps,
    compound="UNKNOWN",
):
    """Urgency index for near-term pit decisions, bounded to [0, 1]."""
    try:
        age_pressure = _clip(_as_float(tyre_life, 0.0) / expected_tyre_life(compound), 0.0, 1.5) / 1.5
        degradation_pressure = _clip(_as_float(degradation, 0.0) / 4.0, 0.0, 1.0)
        rate_pressure = _clip(_as_float(performance_loss_rate, 0.0) / 0.05, 0.0, 1.0)
        pace_pressure = _clip(_as_float(normalized_pace_loss, 0.0) / 0.07, 0.0, 1.0)
        remaining_pressure = _clip(1.0 / max(_as_float(remaining_laps, 1.0), 1.0), 0.0, 1.0)

        pressure = (
            0.35 * _clip(pit_window_score, 0.0, 1.0)
            + 0.20 * age_pressure
            + 0.15 * degradation_pressure
            + 0.15 * rate_pressure
            + 0.10 * pace_pressure
            + 0.05 * remaining_pressure
        )
        return _clip(pressure, 0.0, 1.0)
    except Exception:
        return 0.0


def calculate_current_stint(lap):
    try:
        return int(lap.get("Stint", 1))
    except Exception:
        return 1


def calculate_pit_count(lap):
    return max(0, calculate_current_stint(lap) - 1)


In [ ]:
# ============================================================
# PART 4 - Feature Extraction
# 40 physics-informed features, one-lap-ahead labels, no future features
# ============================================================

DRY_COMPOUNDS = {"SOFT", "MEDIUM", "HARD"}
WET_COMPOUNDS = {"INTERMEDIATE", "WET"}


def _event_value(event, key, default=None):
    try:
        if hasattr(event, "get"):
            return event.get(key, default)
        return getattr(event, key, default)
    except Exception:
        return default


def _load_race_session(year, race):
    session = fastf1.get_session(year, race, "R")
    try:
        session.load(laps=True, telemetry=True, weather=True)
    except TypeError:
        session.load(telemetry=True, weather=True)
    return session


def _compound_flags(compound):
    compound = str(compound).upper()
    soft = int("SOFT" in compound)
    medium = int("MEDIUM" in compound)
    hard = int("HARD" in compound)
    intermediate = int("INTERMEDIATE" in compound)
    wet = int(compound == "WET")
    is_dry = int((soft + medium + hard) > 0)
    is_wet = int((intermediate + wet) > 0)
    return {
        "Soft": soft,
        "Medium": medium,
        "Hard": hard,
        "Intermediate": intermediate,
        "Wet": wet,
        "IsDryTyre": is_dry,
        "IsWetTyre": is_wet,
    }


def _is_pit_lap(lap):
    pit_in = lap.get("PitInTime", pd.NaT)
    pit_out = lap.get("PitOutTime", pd.NaT)
    return (not pd.isna(pit_in)) or (not pd.isna(pit_out))


def _weather_snapshot(weather, lap_start_time):
    defaults = {
        "AirTemp": 0.0,
        "TrackTemp": 0.0,
        "Humidity": 0.0,
        "Rainfall": 0,
    }

    try:
        if weather is None or len(weather) == 0 or pd.isna(lap_start_time):
            return defaults

        weather_frame = weather.copy()
        if "Time" not in weather_frame.columns:
            return defaults

        past_weather = weather_frame[weather_frame["Time"] <= lap_start_time]
        if len(past_weather) == 0:
            row = weather_frame.iloc[0]
        else:
            row = past_weather.iloc[-1]

        rainfall_value = row.get("Rainfall", 0)
        rainfall = int(bool(rainfall_value)) if not pd.isna(rainfall_value) else 0

        return {
            "AirTemp": _clip(row.get("AirTemp", 0.0), -20.0, 60.0),
            "TrackTemp": _clip(row.get("TrackTemp", 0.0), -20.0, 85.0),
            "Humidity": _clip(row.get("Humidity", 0.0), 0.0, 100.0),
            "Rainfall": rainfall,
        }
    except Exception:
        return defaults


def _telemetry_features(lap):
    defaults = {
        "AvgSpeed": 0.0,
        "MaxSpeed": 0.0,
        "SpeedStd": 0.0,
        "ThrottleMean": 0.0,
        "ThrottleStd": 0.0,
        "BrakeMean": 0.0,
        "BrakeRatio": 0.0,
        "RPMMean": 0.0,
        "RPMStd": 0.0,
    }

    try:
        telemetry = lap.get_telemetry()
        if telemetry is None or len(telemetry) == 0:
            return defaults

        speed = pd.to_numeric(telemetry.get("Speed", pd.Series(dtype=float)), errors="coerce")
        throttle = pd.to_numeric(telemetry.get("Throttle", pd.Series(dtype=float)), errors="coerce")
        brake = pd.to_numeric(telemetry.get("Brake", pd.Series(dtype=float)), errors="coerce")
        rpm = pd.to_numeric(telemetry.get("RPM", pd.Series(dtype=float)), errors="coerce")

        max_speed = float(np.nanmax(speed)) if len(speed.dropna()) > 0 else 0.0

        return {
            "AvgSpeed": _clip(safe_mean(speed), 0.0, 380.0),
            "MaxSpeed": _clip(max_speed, 0.0, 380.0),
            "SpeedStd": calculate_speed_std(telemetry),
            "ThrottleMean": _clip(safe_mean(throttle), 0.0, 100.0),
            "ThrottleStd": _clip(safe_std(throttle), 0.0, 60.0),
            "BrakeMean": _clip(safe_mean(brake), 0.0, 100.0),
            "BrakeRatio": calculate_brake_ratio(telemetry),
            "RPMMean": _clip(safe_mean(rpm), 0.0, 20000.0),
            "RPMStd": _clip(safe_std(rpm), 0.0, 8000.0),
        }
    except Exception:
        return defaults


def _valid_lap_time_seconds(laps):
    clean = _clean_lap_history(laps)
    if len(clean) == 0:
        return np.array([], dtype=float)
    return clean["LapTimeSeconds"].to_numpy(dtype=float)


def _best_lap_so_far(lap, laps_driver):
    try:
        lap_num = _as_float(lap.get("LapNumber"), np.nan)
        history = laps_driver[laps_driver["LapNumber"] <= lap_num]
        seconds = _valid_lap_time_seconds(history)
        if len(seconds) == 0:
            return safe_seconds(lap.get("LapTime"), default=0.0)
        return float(np.nanmin(seconds))
    except Exception:
        return safe_seconds(lap.get("LapTime"), default=0.0)


def _estimate_tyre_life(lap, laps_driver):
    tyre_life = lap.get("TyreLife", np.nan)
    if not pd.isna(tyre_life):
        return max(_as_float(tyre_life, 0.0), 0.0)

    try:
        lap_num = _as_float(lap.get("LapNumber"), np.nan)
        stint = lap.get("Stint", np.nan)
        same_stint = laps_driver[
            (laps_driver["LapNumber"] <= lap_num)
            & (laps_driver["Stint"] == stint)
        ]
        return float(max(len(same_stint) - 1, 0))
    except Exception:
        return 0.0


def _will_pit_on_target_lap(laps_driver, current_lap_number, prediction_horizon=1):
    """
    Label = 1 if the driver pits on the target future lap.

    Features for lap t use only information up to lap t. The target checks
    lap t + horizon for PitInTime, or a stint change immediately after that
    target lap. This avoids using the pit-in lap itself as input.
    """
    try:
        target_lap_number = int(current_lap_number + prediction_horizon)
        target = laps_driver[laps_driver["LapNumber"] == target_lap_number]
        if len(target) == 0:
            return np.nan

        target_lap = target.iloc[0]
        if not pd.isna(target_lap.get("PitInTime", pd.NaT)):
            return 1

        after_target = laps_driver[laps_driver["LapNumber"] == target_lap_number + 1]
        if len(after_target) > 0:
            if after_target.iloc[0].get("Stint", np.nan) != target_lap.get("Stint", np.nan):
                return 1

        return 0
    except Exception:
        return np.nan


def extract_features_from_session(
    year,
    race,
    driver,
    round_number=None,
    session=None,
    prediction_horizon=1,
):
    """
    Extract one-lap-ahead pit strategy features for a single driver.

    Output rows are clean racing laps only. Current pit-in and pit-out laps
    are excluded from inputs to prevent direct pit-lane leakage. The label is
    whether the driver pits on the future target lap.
    """
    try:
        if session is None:
            session = _load_race_session(year, race)

        laps_all = session.laps
        weather = getattr(session, "weather_data", pd.DataFrame())

        try:
            laps_driver = (
                laps_all
                .pick_driver(driver)
                .sort_values("LapNumber")
                .reset_index(drop=True)
            )
        except Exception:
            laps_driver = (
                laps_all[laps_all["Driver"] == driver]
                .sort_values("LapNumber")
                .reset_index(drop=True)
            )

        if laps_driver.empty:
            return pd.DataFrame()

        total_laps = int(np.nanmax(laps_all["LapNumber"]))
        event = getattr(session, "event", {})
        race_name = _event_value(event, "EventName", race)
        event_date = _event_value(event, "EventDate", pd.NaT)

        if round_number is None or pd.isna(round_number):
            round_number = _event_value(event, "RoundNumber", np.nan)

        features = []

        for _, lap in laps_driver.iterrows():
            try:
                lap_num = int(_as_float(lap.get("LapNumber"), np.nan))
                if not np.isfinite(lap_num):
                    continue

                if lap_num + prediction_horizon > total_laps:
                    continue

                if pd.isna(lap.get("LapTime", pd.NaT)):
                    continue

                if _is_pit_lap(lap):
                    continue

                will_pit = _will_pit_on_target_lap(
                    laps_driver,
                    lap_num,
                    prediction_horizon=prediction_horizon,
                )
                if pd.isna(will_pit):
                    continue

                stint = calculate_current_stint(lap)
                tyre_life = _estimate_tyre_life(lap, laps_driver)
                compound = str(lap.get("Compound", "UNKNOWN")).upper()
                compound_features = _compound_flags(compound)

                lap_ratio = calculate_lap_ratio(lap_num, total_laps)
                remaining_laps = max(total_laps - lap_num, 0)
                fuel_load = estimate_fuel_mass(lap_num, total_laps)

                telemetry_features = _telemetry_features(lap)
                weather_features = _weather_snapshot(
                    weather,
                    lap.get("LapStartTime", pd.NaT),
                )

                air_temp = weather_features["AirTemp"]
                track_temp = weather_features["TrackTemp"]
                humidity = weather_features["Humidity"]
                rainfall = weather_features["Rainfall"]
                track_air_diff = track_temp - air_temp
                wet_condition_index = float(rainfall) * (humidity / 100.0)

                current_lap_seconds = safe_seconds(lap.get("LapTime"), default=np.nan)
                if not np.isfinite(current_lap_seconds):
                    continue

                best_lap = _best_lap_so_far(lap, laps_driver)
                pace_delta = max(current_lap_seconds - best_lap, 0.0)

                degradation = calculate_tire_degradation(lap, laps_driver, window=3)
                rolling_deg = rolling_degradation(lap, laps_driver, window=5)
                moving_average = calculate_moving_average_lap_time(lap, laps_driver, window=3)
                lap_time_trend = calculate_lap_trend(lap, laps_driver, window=5)

                tire_temperature = tire_temp_proxy(
                    telemetry_features["AvgSpeed"],
                    telemetry_features["ThrottleMean"],
                    telemetry_features["BrakeMean"],
                    track_temp,
                    tyre_life=tyre_life,
                    compound=compound,
                    rainfall=rainfall,
                    humidity=humidity,
                )

                performance_loss_rate = calculate_performance_loss_rate(
                    lap,
                    laps_driver,
                    window=5,
                )

                normalized_pace_loss = calculate_normalized_pace_loss(
                    current_lap_seconds,
                    best_lap,
                )

                is_late_stint = infer_is_late_stint(tyre_life, compound)
                long_run_flag = infer_long_run_flag(tyre_life, compound)

                pit_window_score = calculate_pit_window_score(
                    tyre_life=tyre_life,
                    remaining_laps=remaining_laps,
                    degradation=degradation,
                    pace_loss=normalized_pace_loss,
                    compound=compound,
                    total_laps=total_laps,
                    rainfall=rainfall,
                    humidity=humidity,
                )

                pit_pressure_index = calculate_pit_pressure_index(
                    pit_window_score=pit_window_score,
                    tyre_life=tyre_life,
                    degradation=degradation,
                    performance_loss_rate=performance_loss_rate,
                    normalized_pace_loss=normalized_pace_loss,
                    remaining_laps=remaining_laps,
                    compound=compound,
                )

                features.append({
                    "Year": int(year),
                    "Round": int(round_number) if not pd.isna(round_number) else -1,
                    "Race": race_name,
                    "Driver": driver,
                    "EventDate": event_date,
                    "TotalLaps": total_laps,
                    "Compound": compound,
                    "LapTimeSeconds": current_lap_seconds,

                    "LapNumber": lap_num,
                    "LapRatio": lap_ratio,
                    "Stint": stint,
                    "TyreLife": tyre_life,
                    "RemainingLaps": remaining_laps,
                    "FuelLoad": fuel_load,

                    **compound_features,
                    **telemetry_features,
                    **weather_features,

                    "TrackAirDiff": track_air_diff,
                    "WetConditionIndex": wet_condition_index,
                    "Degradation": degradation,
                    "RollingDegradation": rolling_deg,
                    "PaceDelta": pace_delta,
                    "MovingAverageLapTime": moving_average,
                    "LapTimeTrend": lap_time_trend,
                    "TireTemperatureProxy": tire_temperature,
                    "PitWindowScore": pit_window_score,
                    "IsLateStint": is_late_stint,
                    "LongRunFlag": long_run_flag,
                    "PerformanceLossRate": performance_loss_rate,
                    "NormalizedPaceLoss": normalized_pace_loss,
                    "PitPressureIndex": pit_pressure_index,
                    "WillPit": int(will_pit),
                })

            except Exception:
                continue

        return pd.DataFrame(features)

    except Exception as e:
        print(f"Feature extraction failed: {year} {race} {driver}: {e}")
        return pd.DataFrame()


In [ ]:
# ============================================================
# PART 5 - Build Full Dataset
# Session-level loading, driver-level feature extraction
# ============================================================

all_data = []
session_summaries = []

success_count = 0
empty_count = 0
error_count = 0

if "all_sessions" not in globals() or len(all_sessions) == 0:
    raise ValueError("all_sessions is empty. Run PART 2 first.")

for session_meta in all_sessions:
    year = int(session_meta["year"])
    round_number = int(session_meta.get("round", -1))
    race = session_meta["race"]
    drivers = session_meta.get("drivers", [])

    print(f"\nLoading race once: {year} R{round_number} {race}")

    try:
        session = _load_race_session(year, race)
    except Exception as e:
        print(f"Session load error: {year} {race}: {e}")
        error_count += len(drivers)
        continue

    race_rows = 0

    for driver in drivers:
        print(f"  Extracting {driver}")

        try:
            df_driver = extract_features_from_session(
                year=year,
                race=race,
                driver=driver,
                round_number=round_number,
                session=session,
                prediction_horizon=1,
            )

            if df_driver.empty:
                empty_count += 1
                continue

            all_data.append(df_driver)
            race_rows += len(df_driver)
            success_count += 1

        except Exception as e:
            print(f"  ERROR {year} {race} {driver}: {e}")
            error_count += 1

    session_summaries.append({
        "Year": year,
        "Round": round_number,
        "Race": race,
        "DriversProcessed": len(drivers),
        "Rows": race_rows,
    })

    del session
    gc.collect()

if len(all_data) == 0:
    raise ValueError("No valid data extracted.")

full_df = (
    pd.concat(all_data, ignore_index=True)
    .drop_duplicates(subset=["Year", "Round", "Race", "Driver", "LapNumber"])
    .sort_values(["Year", "Round", "Race", "Driver", "LapNumber"])
    .reset_index(drop=True)
)

session_summary_df = pd.DataFrame(session_summaries)

print("\n========== DATASET BUILD SUMMARY ==========")
print("Successful driver datasets:", success_count)
print("Empty driver datasets:", empty_count)
print("Errors:", error_count)
print("Full dataset shape:", full_df.shape)
print("\nWillPit distribution:")
print(full_df["WillPit"].value_counts(dropna=False))
print("\nSession summary:")
print(session_summary_df.tail())

del all_data
gc.collect()


core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...



Loading race once: 2019 R1 Australian Grand Prix


core        WARNING 	Driver 77 completed the race distance 00:00.387000 before the recorded end of the session.
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['77', '44', '33', '5', '16', '20', '27', '7', '18', '26', '10', '4', '11', '23', '99', '63', '88', '8', '3', '55']


  Extracting ALB
  Extracting BOT
  Extracting GAS
  Extracting GIO
  Extracting GRO
  Extracting HAM
  Extracting HUL
  Extracting KUB
  Extracting KVY
  Extracting LEC
  Extracting MAG
  Extracting NOR
  Extracting PER
  Extracting RAI
  Extracting RIC
  Extracting RUS
  Extracting SAI
  Extracting STR
  Extracting VER
  Extracting VET

Loading race once: 2019 R2 Bahrain Grand Prix


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 44 completed the race distance 00:00.070000 before the recorded end of the session.
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['44', '77', '16', '33', '5', '4', '7', '10', '23', '11', '99', '26

...

core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info



Loading race once: 2024 R24 Abu Dhabi Grand Prix


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '55', '16', '44', '63', '1', '10', '27', '14', '81', '23', '22', '24', '18', '61', '20', '30', '77', '43', '11']


  Extracting ALB
  Extracting ALO
  Extracting BOT
  Extracting COL
  Extracting DOO
  Extracting GAS
  Extracting HAM
  Extracting HUL
  Extracting LAW
  Extracting LEC
  Extracting MAG
  Extracting NOR
  Extracting PER
  Extracting PIA
  Extracting RUS
  Extracting SAI
  Extracting STR
  Extracting TSU
  Extracting VER
  Extracting ZHO

========== DATASET BUILD SUMMARY ==========
Successful driver datasets: 2606
Empty driver datasets: 93
Errors: 0
Full dataset shape: (133082, 49)

WillPit distribution:
WillPit
0    128576
1      4506
Name: count, dtype: int64

Session summary:
     Year  Round                    Race  DriversProcessed  Rows
130  2024     20  Mexico City Grand Prix                20  1152
131  2024     21    São Paulo Grand Prix                20  1050
132  2024     22    Las Vegas Grand Prix                20   840
133  2024     23        Qatar Grand Prix                20   832
134  2024     24    Abu Dhabi Grand Prix                20   961


0

In [ ]:
# ============================================================
# PART 6 - Data Cleaning
# Physics bounds, dtype normalization, label validation
# ============================================================

full_df = full_df.copy()
full_df.replace([np.inf, -np.inf], np.nan, inplace=True)

ID_COLUMNS = ["Year", "Round", "Race", "Driver", "LapNumber"]
NON_NUMERIC_COLUMNS = ["Race", "Driver", "Compound", "EventDate"]

for column in full_df.columns:
    if column not in NON_NUMERIC_COLUMNS:
        full_df[column] = pd.to_numeric(full_df[column], errors="coerce")

full_df = full_df.dropna(subset=["Year", "Round", "Race", "Driver", "LapNumber", "WillPit"])
full_df = full_df[full_df["WillPit"].isin([0, 1])]

DEFAULT_FILL_VALUES = {
    "LapRatio": 0.0,
    "Stint": 1.0,
    "TyreLife": 0.0,
    "RemainingLaps": 0.0,
    "FuelLoad": FIA_MAX_RACE_FUEL_KG,
    "AvgSpeed": 0.0,
    "MaxSpeed": 0.0,
    "SpeedStd": 0.0,
    "ThrottleMean": 0.0,
    "ThrottleStd": 0.0,
    "BrakeMean": 0.0,
    "BrakeRatio": 0.0,
    "RPMMean": 0.0,
    "RPMStd": 0.0,
    "AirTemp": 0.0,
    "TrackTemp": 0.0,
    "Humidity": 0.0,
    "Rainfall": 0.0,
    "TrackAirDiff": 0.0,
    "WetConditionIndex": 0.0,
    "Degradation": 0.0,
    "RollingDegradation": 0.0,
    "PaceDelta": 0.0,
    "MovingAverageLapTime": 0.0,
    "LapTimeTrend": 0.0,
    "TireTemperatureProxy": 0.0,
    "PitWindowScore": 0.0,
    "IsLateStint": 0.0,
    "LongRunFlag": 0.0,
    "PerformanceLossRate": 0.0,
    "NormalizedPaceLoss": 0.0,
    "PitPressureIndex": 0.0,
}

for column, value in DEFAULT_FILL_VALUES.items():
    if column in full_df.columns:
        full_df[column] = full_df[column].fillna(value)

BINARY_COLUMNS = [
    "Soft",
    "Medium",
    "Hard",
    "Intermediate",
    "Wet",
    "IsDryTyre",
    "IsWetTyre",
    "Rainfall",
    "IsLateStint",
    "LongRunFlag",
    "WillPit",
]

for column in BINARY_COLUMNS:
    if column in full_df.columns:
        full_df[column] = full_df[column].fillna(0).round().clip(0, 1).astype(int)

CLIP_BOUNDS = {
    "LapNumber": (1, 120),
    "LapRatio": (0, 1),
    "Stint": (1, 8),
    "TyreLife": (0, 90),
    "RemainingLaps": (0, 120),
    "FuelLoad": (0, FIA_MAX_RACE_FUEL_KG),
    "AvgSpeed": (0, 380),
    "MaxSpeed": (0, 380),
    "SpeedStd": (0, 140),
    "ThrottleMean": (0, 100),
    "ThrottleStd": (0, 60),
    "BrakeMean": (0, 100),
    "BrakeRatio": (0, 1),
    "RPMMean": (0, 20000),
    "RPMStd": (0, 8000),
    "AirTemp": (-20, 60),
    "TrackTemp": (-20, 85),
    "Humidity": (0, 100),
    "TrackAirDiff": (-40, 80),
    "WetConditionIndex": (0, 1),
    "Degradation": (-10, 20),
    "RollingDegradation": (-5, 5),
    "PaceDelta": (0, 120),
    "MovingAverageLapTime": (0, 300),
    "LapTimeTrend": (-5, 5),
    "TireTemperatureProxy": (0, 160),
    "PitWindowScore": (0, 1),
    "PerformanceLossRate": (-0.03, 0.12),
    "NormalizedPaceLoss": (0, 0.35),
    "PitPressureIndex": (0, 1),
}

for column, (lower, upper) in CLIP_BOUNDS.items():
    if column in full_df.columns:
        full_df[column] = full_df[column].clip(lower, upper)

full_df = full_df[
    (full_df["AvgSpeed"] > 50)
    & (full_df["MaxSpeed"] < 380)
    & (full_df["TyreLife"] >= 0)
    & (full_df["FuelLoad"] >= 0)
].copy()

full_df = (
    full_df
    .drop_duplicates(subset=["Year", "Round", "Race", "Driver", "LapNumber"])
    .sort_values(["Year", "Round", "Race", "Driver", "LapNumber"])
    .reset_index(drop=True)
)

print("\n========== CLEAN DATASET ==========")
print("Shape:", full_df.shape)
print("\nWillPit distribution:")
print(full_df["WillPit"].value_counts())
print("\nWillPit ratio:")
print(full_df["WillPit"].value_counts(normalize=True))
print("\nRemaining missing values:")
print(full_df.isnull().sum()[full_df.isnull().sum() > 0])



========== CLEAN DATASET ==========
Shape: (133047, 49)

WillPit distribution:
WillPit
0    128541
1      4506
Name: count, dtype: int64

WillPit ratio:
WillPit
0    0.966132
1    0.033868
Name: proportion, dtype: float64

Remaining missing values:
Series([], dtype: int64)


In [ ]:
# ============================================================
# PART 7 - Feature Columns
# 40 research features grouped by racing mechanism
# ============================================================

FEATURE_GROUPS = {
    "RaceState": [
        "LapNumber",
        "LapRatio",
        "Stint",
        "TyreLife",
        "RemainingLaps",
        "FuelLoad",
    ],
    "TireCompound": [
        "Soft",
        "Medium",
        "Hard",
        "Intermediate",
        "Wet",
        "IsDryTyre",
        "IsWetTyre",
    ],
    "Telemetry": [
        "AvgSpeed",
        "MaxSpeed",
        "SpeedStd",
        "ThrottleMean",
        "ThrottleStd",
        "BrakeMean",
        "BrakeRatio",
        "RPMMean",
        "RPMStd",
    ],
    "Weather": [
        "AirTemp",
        "TrackTemp",
        "Humidity",
        "Rainfall",
        "TrackAirDiff",
        "WetConditionIndex",
    ],
    "TirePhysics": [
        "Degradation",
        "RollingDegradation",
        "PaceDelta",
        "MovingAverageLapTime",
        "LapTimeTrend",
        "TireTemperatureProxy",
    ],
    "StrategySignal": [
        "PitWindowScore",
        "IsLateStint",
        "LongRunFlag",
        "PerformanceLossRate",
        "NormalizedPaceLoss",
        "PitPressureIndex",
    ],
}

FEATURE_COLUMNS = [
    feature
    for group_features in FEATURE_GROUPS.values()
    for feature in group_features
]

TARGET_COLUMN = "WillPit"
GROUP_COLUMNS = ["Year", "Round", "Race", "Driver"]
SEQUENCE_METADATA_COLUMNS = GROUP_COLUMNS + ["LapNumber", "Stint", "Compound"]

missing_features = [col for col in FEATURE_COLUMNS if col not in full_df.columns]
if missing_features:
    raise ValueError(f"Missing features: {missing_features}")

if TARGET_COLUMN not in full_df.columns:
    raise ValueError(f"{TARGET_COLUMN} not found")

if len(FEATURE_COLUMNS) != 40:
    raise ValueError(f"Expected 40 features, found {len(FEATURE_COLUMNS)}")

print("\n========== FEATURE VALIDATION ==========")
for group_name, group_features in FEATURE_GROUPS.items():
    print(f"{group_name}: {len(group_features)}")
print("Feature Count:", len(FEATURE_COLUMNS))
print("Target:", TARGET_COLUMN)
print("Validation passed.")



========== FEATURE VALIDATION ==========
RaceState: 6
TireCompound: 7
Telemetry: 9
Weather: 6
TirePhysics: 6
StrategySignal: 6
Feature Count: 40
Target: WillPit
Validation passed.


In [ ]:
# ============================================================
# PART 8 - Sliding Window
# Race/driver leakage prevention and train-only scaling
# ============================================================

WINDOW_SIZE = 5
VALIDATION_RACE_FRACTION = 0.20


def _race_key_frame(df):
    return (
        df[["Year", "Round", "Race"]]
        .drop_duplicates()
        .sort_values(["Year", "Round", "Race"])
        .reset_index(drop=True)
    )


def _assign_chronological_race_split(df, validation_fraction=0.20):
    races = _race_key_frame(df)
    if len(races) < 2:
        raise ValueError("Need at least two races for race-level train/validation split.")

    n_val = max(1, int(math.ceil(len(races) * validation_fraction)))
    val_races = races.tail(n_val).copy()
    val_keys = set(map(tuple, val_races[["Year", "Round", "Race"]].to_numpy()))

    split = []
    for _, row in df[["Year", "Round", "Race"]].iterrows():
        key = (row["Year"], row["Round"], row["Race"])
        split.append("val" if key in val_keys else "train")

    return pd.Series(split, index=df.index), val_races


def _build_sequences_from_frame(df, feature_columns, target_column, window_size):
    X_sequences = []
    y_sequences = []
    metadata = []

    grouped = df.groupby(["Year", "Round", "Race", "Driver"], sort=False)

    for (year, round_number, race, driver), group in grouped:
        group = (
            group
            .sort_values("LapNumber")
            .reset_index(drop=True)
        )

        if len(group) < window_size:
            continue

        for start_idx in range(0, len(group) - window_size + 1):
            window = group.iloc[start_idx:start_idx + window_size]

            if window["Stint"].nunique() != 1:
                continue

            if not window["LapNumber"].is_monotonic_increasing:
                continue

            X_sequences.append(window[feature_columns].to_numpy(dtype=np.float32))
            y_sequences.append(float(window[target_column].iloc[-1]))

            end_row = window.iloc[-1]
            metadata.append({
                "Year": year,
                "Round": round_number,
                "Race": race,
                "Driver": driver,
                "Stint": int(end_row["Stint"]),
                "LapNumber": int(end_row["LapNumber"]),
                "Compound": end_row.get("Compound", "UNKNOWN"),
                "WindowStartLap": int(window["LapNumber"].iloc[0]),
                "WindowEndLap": int(end_row["LapNumber"]),
                "WillPit": int(end_row[target_column]),
            })

    X_sequences = np.asarray(X_sequences, dtype=np.float32)
    y_sequences = np.asarray(y_sequences, dtype=np.float32)
    metadata = pd.DataFrame(metadata)

    return X_sequences, y_sequences, metadata


full_df = full_df.sort_values(["Year", "Round", "Race", "Driver", "LapNumber"]).reset_index(drop=True)
full_df["Split"], validation_races = _assign_chronological_race_split(
    full_df,
    validation_fraction=VALIDATION_RACE_FRACTION,
)

train_df = full_df[full_df["Split"] == "train"].copy()
val_df = full_df[full_df["Split"] == "val"].copy()

X_train_raw, y_train, train_sequence_metadata = _build_sequences_from_frame(
    train_df,
    FEATURE_COLUMNS,
    TARGET_COLUMN,
    WINDOW_SIZE,
)

X_val_raw, y_val, val_sequence_metadata = _build_sequences_from_frame(
    val_df,
    FEATURE_COLUMNS,
    TARGET_COLUMN,
    WINDOW_SIZE,
)

if len(X_train_raw) == 0 or len(X_val_raw) == 0:
    raise ValueError("No train or validation sequences were created. Reduce WINDOW_SIZE or check data coverage.")

scaler = StandardScaler()
train_2d = X_train_raw.reshape(-1, len(FEATURE_COLUMNS))
val_2d = X_val_raw.reshape(-1, len(FEATURE_COLUMNS))

scaler.fit(train_2d)

X_train = scaler.transform(train_2d).reshape(X_train_raw.shape).astype(np.float32)
X_val = scaler.transform(val_2d).reshape(X_val_raw.shape).astype(np.float32)

y_train = y_train.astype(np.float32)
y_val = y_val.astype(np.float32)

X_sequences = np.concatenate([X_train, X_val], axis=0)
y_sequences = np.concatenate([y_train, y_val], axis=0)
sequence_metadata = pd.concat(
    [
        train_sequence_metadata.assign(Split="train"),
        val_sequence_metadata.assign(Split="val"),
    ],
    ignore_index=True,
)

print("\n========== SLIDING WINDOW SUMMARY ==========")
print("Window size:", WINDOW_SIZE)
print("Train sequences:", X_train.shape, y_train.shape)
print("Validation sequences:", X_val.shape, y_val.shape)
print("Validation races:")
print(validation_races)
print("\nTrain label ratio:", float(np.mean(y_train)))
print("Validation label ratio:", float(np.mean(y_val)))



========== SLIDING WINDOW SUMMARY ==========
Window size: 5
Train sequences: (84654, 5, 40) (84654,)
Validation sequences: (22149, 5, 40) (22149,)
Validation races:
     Year  Round                       Race
106  2023     21       Las Vegas Grand Prix
107  2023     22       Abu Dhabi Grand Prix
108  2024      0       Singapore Grand Prix
109  2024      1         Bahrain Grand Prix
110  2024      2   Saudi Arabian Grand Prix
111  2024      3      Australian Grand Prix
112  2024      4        Japanese Grand Prix
113  2024      5         Chinese Grand Prix
114  2024      6           Miami Grand Prix
115  2024      7  Emilia Romagna Grand Prix
116  2024      8          Monaco Grand Prix
117  2024      9        Canadian Grand Prix
118  2024     10         Spanish Grand Prix
119  2024     11        Austrian Grand Prix
120  2024     12         British Grand Prix
121  2024     13       Hungarian Grand Prix
122  2024     14         Belgian Grand Prix
123  2024     15           Dutch Grand Pri

In [ ]:
# ============================================================
# PART 9 - LSTM Dataset Class
# ============================================================

class F1PitStopDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.as_tensor(X, dtype=torch.float32)
        self.y = torch.as_tensor(y, dtype=torch.float32)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


F1Dataset = F1PitStopDataset

BATCH_SIZE = 64

train_dataset = F1PitStopDataset(X_train, y_train)
val_dataset = F1PitStopDataset(X_val, y_val)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))


Train batches: 1323
Validation batches: 347


In [ ]:
# ============================================================
# PART 10 - BiLSTM + Attention Model
# Logit output for BCEWithLogitsLoss
# ============================================================

class TemporalAttention(nn.Module):

    def __init__(self, input_dim, attention_dim=128, dropout=0.2):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(input_dim, attention_dim),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(attention_dim, 1),
        )

    def forward(self, sequence_output):
        attention_logits = self.score(sequence_output).squeeze(-1)
        attention_weights = torch.softmax(attention_logits, dim=1)
        context = torch.sum(sequence_output * attention_weights.unsqueeze(-1), dim=1)
        return context, attention_weights


class F1StrategyModel(nn.Module):

    def __init__(
        self,
        input_size=None,
        hidden_size=128,
        num_layers=2,
        dropout=0.35,
    ):
        super().__init__()

        if input_size is None:
            input_size = len(FEATURE_COLUMNS)

        self.input_size = input_size
        self.hidden_size = hidden_size

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
            bidirectional=True,
        )

        lstm_dim = hidden_size * 2
        self.sequence_norm = nn.LayerNorm(lstm_dim)
        self.attention = TemporalAttention(
            input_dim=lstm_dim,
            attention_dim=hidden_size,
            dropout=dropout,
        )

        self.head = nn.Sequential(
            nn.LayerNorm(lstm_dim),
            nn.Linear(lstm_dim, 64),
            nn.GELU(),
            nn.Dropout(0.30),
            nn.Linear(64, 32),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(32, 1),
        )

    def forward(self, x, return_attention=False):
        sequence_output, _ = self.lstm(x)
        sequence_output = self.sequence_norm(sequence_output)
        context, attention_weights = self.attention(sequence_output)
        logits = self.head(context).squeeze(-1)

        if return_attention:
            return logits, attention_weights

        return logits


model = F1StrategyModel(input_size=len(FEATURE_COLUMNS)).to(device)

print(model)


F1StrategyModel(
  (lstm): LSTM(40, 128, num_layers=2, batch_first=True, dropout=0.35, bidirectional=True)
  (sequence_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (attention): TemporalAttention(
    (score): Sequential(
      (0): Linear(in_features=256, out_features=128, bias=True)
      (1): Tanh()
      (2): Dropout(p=0.35, inplace=False)
      (3): Linear(in_features=128, out_features=1, bias=True)
    )
  )
  (head): Sequential(
    (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=256, out_features=64, bias=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): GELU(approximate='none')
    (6): Dropout(p=0.2, inplace=False)
    (7): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [ ]:
# ============================================================
# PART 11 - Weighted BCE Loss
# Train-split pos_weight for severe class imbalance
# ============================================================

positive_count = float(np.sum(y_train == 1))
negative_count = float(np.sum(y_train == 0))

if positive_count == 0:
    pos_weight_value = 1.0
else:
    pos_weight_value = negative_count / positive_count

pos_weight = torch.tensor(
    [pos_weight_value],
    dtype=torch.float32,
    device=device,
)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

print("Train positives:", int(positive_count))
print("Train negatives:", int(negative_count))
print("pos_weight:", float(pos_weight.item()))


Train positives: 3307
Train negatives: 81347
pos_weight: 24.598426818847656


In [ ]:
# ============================================================
# PART 12 - Training Configuration
# ============================================================

from torch.cuda.amp import autocast
from torch.cuda.amp import GradScaler

AMP_ENABLED = device.type == "cuda"
scaler_amp = GradScaler(enabled=AMP_ENABLED)

EPOCHS = 30
PATIENCE = 5
GRAD_CLIP_NORM = 1.0
THRESHOLD = 0.5

best_val_loss = float("inf")
patience_counter = 0

CHECKPOINT_PATH = "best_f1_strategy_model_attention.pth"

print("AMP enabled:", AMP_ENABLED)
print("Checkpoint:", CHECKPOINT_PATH)


AMP enabled: True
Checkpoint: best_f1_strategy_model_attention.pth


In [ ]:
# ============================================================
# PART 13 - Evaluation Helpers
# ============================================================

def evaluate_strategy_model(model, data_loader, criterion=None, threshold=0.5):
    model.eval()

    total_loss = 0.0
    batch_count = 0
    all_probs = []
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)

            if criterion is not None:
                loss = criterion(logits, y_batch)
                total_loss += loss.item()
                batch_count += 1

            probs = torch.sigmoid(logits)
            preds = (probs >= threshold).int()

            all_probs.extend(probs.detach().cpu().numpy())
            all_preds.extend(preds.detach().cpu().numpy())
            all_targets.extend(y_batch.detach().cpu().numpy())

    metrics = {
        "loss": total_loss / max(batch_count, 1),
        "accuracy": accuracy_score(all_targets, all_preds),
        "precision": precision_score(all_targets, all_preds, zero_division=0),
        "recall": recall_score(all_targets, all_preds, zero_division=0),
        "f1": f1_score(all_targets, all_preds, zero_division=0),
    }

    return metrics, np.asarray(all_probs), np.asarray(all_preds), np.asarray(all_targets)


In [ ]:
# ============================================================
# PART 14 - Training Loop
# ============================================================

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    train_batches = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=AMP_ENABLED):
            logits = model(X_batch)
            loss = criterion(logits, y_batch)

        scaler_amp.scale(loss).backward()
        scaler_amp.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler_amp.step(optimizer)
        scaler_amp.update()

        train_loss += loss.item()
        train_batches += 1

    avg_train_loss = train_loss / max(train_batches, 1)
    val_metrics, _, _, _ = evaluate_strategy_model(
        model,
        val_loader,
        criterion=criterion,
        threshold=THRESHOLD,
    )

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"Train Loss {avg_train_loss:.4f} | "
        f"Val Loss {val_metrics['loss']:.4f} | "
        f"Acc {val_metrics['accuracy']:.4f} | "
        f"F1 {val_metrics['f1']:.4f} | "
        f"P {val_metrics['precision']:.4f} | "
        f"R {val_metrics['recall']:.4f}"
    )

    if val_metrics["loss"] < best_val_loss:
        best_val_loss = val_metrics["loss"]
        patience_counter = 0
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print("Model saved.")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping.")
            break


Epoch 01/30 | Train Loss 1.2269 | Val Loss 1.2792 | Acc 0.8405 | F1 0.1976 | P 0.1209 | R 0.5397
Model saved.
Epoch 02/30 | Train Loss 1.1690 | Val Loss 1.3340 | Acc 0.8521 | F1 0.2026 | P 0.1260 | R 0.5161
Epoch 03/30 | Train Loss 1.1320 | Val Loss 1.0826 | Acc 0.7853 | F1 0.1873 | P 0.1086 | R 0.6799
Model saved.
Epoch 04/30 | Train Loss 1.1110 | Val Loss 1.1393 | Acc 0.7783 | F1 0.1792 | P 0.1036 | R 0.6650
Epoch 05/30 | Train Loss 1.1072 | Val Loss 1.2003 | Acc 0.8282 | F1 0.1954 | P 0.1178 | R 0.5732
Epoch 06/30 | Train Loss 1.0920 | Val Loss 1.5207 | Acc 0.8281 | F1 0.1950 | P 0.1175 | R 0.5720
Epoch 07/30 | Train Loss 1.0756 | Val Loss 1.4518 | Acc 0.8199 | F1 0.1943 | P 0.1160 | R 0.5968
Epoch 08/30 | Train Loss 1.0537 | Val Loss 1.2268 | Acc 0.7650 | F1 0.1684 | P 0.0967 | R 0.6538
Early stopping.


In [ ]:
# ============================================================
# PART 15 - Real Prediction Engine
# ============================================================

if os.path.exists(CHECKPOINT_PATH):
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))

model.eval()


def predict_next_pit_probability(recent_dataframe, return_attention=False):
    """
    Predict probability of pitting on the next lap.

    recent_dataframe must contain the same 40 feature columns. If it contains
    more than WINDOW_SIZE rows, only the latest WINDOW_SIZE laps are used.
    """
    if len(recent_dataframe) < WINDOW_SIZE:
        raise ValueError(f"Need at least {WINDOW_SIZE} recent laps.")

    recent = recent_dataframe.tail(WINDOW_SIZE).copy()
    missing = [col for col in FEATURE_COLUMNS if col not in recent.columns]
    if missing:
        raise ValueError(f"Missing inference features: {missing}")

    recent_features = (
        recent[FEATURE_COLUMNS]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .to_numpy(dtype=np.float32)
    )

    scaled = scaler.transform(recent_features).astype(np.float32)
    sequence = torch.as_tensor(scaled, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        if return_attention:
            logits, attention_weights = model(sequence, return_attention=True)
        else:
            logits = model(sequence)
            attention_weights = None

        probability = torch.sigmoid(logits).item()

    if return_attention:
        return probability, attention_weights.squeeze(0).detach().cpu().numpy()

    return probability
